In [1]:
%matplotlib qt

In [2]:
# from turtle import pos

import matplotlib

# matplotlib.use("Agg")  # for mac users

#print(matplotlib.get_backend())

import sys

import matplotlib as mpl

sys.path.append("/home/weipeng/CODES/Smilei")
import happi
import matplotlib.pyplot as plt
import numpy as np

jetcmap = plt.cm.get_cmap(
    "jet", 9
)  # generate a jet map with 10 values "rainbow", "jet", YlOrRd
jet_vals = jetcmap(np.arange(9))  # extract those values as an array
jet_vals[0] = [1.0, 1, 1.0, 1]  # change the first value
jet_vals[8] = [0.0, 0, 0.0, 1]  # change the first value
newcmap = mpl.colors.LinearSegmentedColormap.from_list("mine", jet_vals)

from matplotlib import font_manager

font_dirs = ["/Users/yao/Documents/Calibri and Cambria Fonts/"]
font_files = font_manager.findSystemFonts(fontpaths=font_dirs)

for font_file in font_files:
   font_manager.fontManager.addfont(font_file)

# set font
plt.rcParams["font.family"] = "Calibri"

plt.rc("text", usetex=False)
plt.rc("xtick", labelsize=14)
plt.rc("ytick", labelsize=14)
plt.rc("axes", labelsize=14)
plt.rc("legend", fontsize=12)

import scienceplots as splt

plt.style.use(["nature", "notebook", "grid", "high-vis"])

/var/folders/2t/97rc3fl92tg15k2l_4sk5hsh0000gn/T/ipykernel_11527/4764761.py:18: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  jetcmap = plt.cm.get_cmap(


In [3]:
wkdir = [
    "/Users/yao/Smilei/titan/",
]


S0 = happi.Open(
    wkdir[0], reference_angular_frequency_SI=2.0 * np.pi * 3e8 / (1.0 * 1e-6)
)

Loaded simulation '/Users/yao/Smilei/titan/'
Scanning for Scalar diagnostics
Scanning for Field diagnostics
Scanning for Probe diagnostics
Scanning for ParticleBinning diagnostics
Scanning for RadiationSpectrum diagnostics
Scanning for Performance diagnostics
Scanning for Screen diagnostics
Scanning for Tracked particle diagnostics
Scanning for new particle diagnostics


In [14]:
S0.Field(0,'Ey', units=["um", "fs", "V/m"]).slide()

Field diagnostic #0: Ey


In [5]:
import numpy as np
import matplotlib.pyplot as plt

def fwhm_from_curve(t, y):
    """
    FWHM for a single-peaked curve y(t) using linear interpolation at half max.
    Returns: fwhm, t_left, t_right, y_half
    """
    t = np.asarray(t)
    y = np.asarray(y)

    # sort just in case
    idx = np.argsort(t)
    t = t[idx]
    y = y[idx]

    imax = np.argmax(y)
    y_max = y[imax]
    y_half = 0.5 * y_max

    # left crossing
    yl = y[:imax+1]
    tl = t[:imax+1]
    left_idx = np.where((yl[:-1] < y_half) & (yl[1:] >= y_half))[0]
    if len(left_idx) == 0:
        raise ValueError("No left half-maximum crossing found.")
    i = left_idx[-1]
    tL = tl[i] + (y_half - yl[i]) * (tl[i+1] - tl[i]) / (yl[i+1] - yl[i])

    # right crossing
    yr = y[imax:]
    tr = t[imax:]
    right_idx = np.where((yr[:-1] >= y_half) & (yr[1:] < y_half))[0]
    if len(right_idx) == 0:
        raise ValueError("No right half-maximum crossing found.")
    j = right_idx[0]
    tR = tr[j] + (y_half - yr[j]) * (tr[j+1] - tr[j]) / (yr[j+1] - yr[j])

    return (tR - tL), tL, tR, y_half


# -----------------------
# Your data construction
# -----------------------
Ey_field = S0.Field(0, 'Ey', units=["um", "fs", "V/m"])
t_arr = Ey_field.getTimes()

tstop = t_arr[-1]
tstep = t_arr[1] - t_arr[0]

laser = S0.namelist.Laser[0]

times = np.arange(0., tstop, tstep)   # fs
laser_profile = np.array([laser.time_envelope(t) for t in times])

# Use relative intensity ~ |E|^2
Irel = laser_profile**2

# -----------------------
# FWHM + plot annotation
# -----------------------
fwhm_fs, tL, tR, I_half = fwhm_from_curve(times, Irel)

plt.figure(figsize=(7,4))
plt.plot(times, Irel, lw=3)
plt.axhline(I_half, ls="--", color="k", lw=2, alpha=0.8)
plt.axvline(tL, color="k", ls=":", lw=1.5)
plt.axvline(tR, color="k", ls=":", lw=1.5)

# double arrow + label
y_annot = I_half * 1.05
plt.annotate(
    "", xy=(tL, y_annot), xytext=(tR, y_annot),
    arrowprops=dict(arrowstyle="<->", lw=2, color="k")
)
plt.text(
    0.5*(tL+tR), y_annot*1.03,
    f"FWHM = {fwhm_fs:.0f} fs",
    ha="center", va="bottom"
)

plt.xlabel("time (fs)")
plt.ylabel(r"relative intensity $\propto |E|^2$ (a.u.)")
plt.ylim(bottom=0)
plt.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.xlim(times[0], times[-1])
plt.show()

print(f"FWHM = {fwhm_fs:.2f} fs  (tL={tL:.2f} fs, tR={tR:.2f} fs)")
plt.savefig("/Users/yao/Desktop/laser_profile_fwhm.png", dpi=300)

FWHM = 500.54 fs  (tL=810.96 fs, tR=1311.50 fs)


In [74]:
S0.ParticleBinning(0, units=["um", "fs", "V/m","kg m^-3"],
                   data_log=True,
                   cmap=newcmap,
                   vmin=-6, vmax=-2,
                   ).slide()


#0 - Number density of species # 0
    x from 0 to 4398.23 in 500 steps 
    px from -5000 to 5000 in 500 steps 

The value in each bin is the sum of the `deposited_quantity` divided by the bin size



In [75]:
S0.ParticleBinning(1, units=["um", "fs", "V/m","kg m^-3"],
                   data_log=True,
                   cmap=newcmap,
                   vmin=-5, vmax=-1,
                   ).slide()


#1 - Number density of species # 1
    x from 0 to 4398.23 in 500 steps 
    px from -2000 to 2000 in 500 steps 

The value in each bin is the sum of the `deposited_quantity` divided by the bin size



In [76]:
S0.ParticleBinning(2, units=["um", "fs", "V/m","kg m^-3"],
                   data_log=True,
                   cmap=newcmap,
                   vmin=-4, vmax=0,
                   ).slide()


#2 - Number density of species # 2
    x from 0 to 4398.23 in 500 steps 
    px from -200 to 200 in 500 steps 

The value in each bin is the sum of the `deposited_quantity` divided by the bin size



In [79]:
S0.ParticleBinning(5, units=["um", "fs", "V/m","MeV"],
                   data_log=True,
                   cmap=newcmap,
                #    vmin=-3, vmax=1,
                   ).slide()


#5 - Number density of species # 2
    ekin from 0 to 195.695 in 400 steps 

The value in each bin is the sum of the `deposited_quantity` divided by the bin size and by grid_length[0]



In [81]:
Ey = S0.Field(0,'Ey', units=["um", "fs"])#.slide()
Re = S0.Field(0,'Rho_eon', units=["um", "fs"])#.slide()

happi.multiSlide(Ey, Re)




In [19]:
S0.Field(0,'Rho_ion_H1', units=["um", "fs", "V/m","kg m^-3"]).slide()

Field diagnostic #0: Rho_ion_H1


In [20]:
S0.Field(0,'Rho_eon', units=["um", "fs", "V/m","kg m^-3"]).slide()

Field diagnostic #0: Rho_eon


In [35]:
tstop = S0.namelist.Main.simulation_time # simulation final time /

In [36]:
5654.8667764616275 / tstop

1.0

In [47]:
S0.namelist.Main.simulation_time / S0.namelist.Main.timestep / 6.28

2413.677505866577

In [46]:
S0.namelist.Main.timestep

0.37306412761378793